# Examining the effect of cloud masking algorithms on remote sensing data accuracy and processing to improve water quality in Sandusky Bay, Ohio

Cloud removal is crucial to the improvement of satellite data quality, which can be used to detect spectral signatures of water bodies for the purpose of accurately predicting harmful algal bloom (HAB) events. Specifically, this project seeks to compare the cloud masks and resulting cloud-free composites of popular cloud-masking algorithms against labeled reference data used as ground truth. Each algorithm is implemented using the Harmonized Sentinel-2 Level-2A surface reflectance collection and tested using a total of 15 scenes with varying levels of cloudiness over the Sandusky Bay region of Ohio. Each algorithm masks both opaque and cirrus cloud types, yet only one of them explicitly detects cloud shadows (Cloud Score+). The aim is to determine which algorithm produces the best results for this region of interest and potentially other freshwater environments at risk of environmental harm. In a freshwater environment, cloud shadows and thin clouds are critical, especially as small water bodies are highly sensitive to misclassification. Algorithm selection was based on alignment with objectives, feasibility of implementation, and the results of research papers.

## Initialization

### Imports and installations

In [ ]:
import getpass
import time
import json
from pathlib import Path
import logging

import ee
import ee.batch
import yaml  # For reading configuration files.
import requests
import folium  # Visualization (maps)

try:
    from google.colab import drive  # For mounting Google Drive in Colab.
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

# ----------------------------------
# For use in image creation for IRIS.
# ----------------------------------
%pip install -U 'imagecodecs[all]'
import imagecodecs
import tifffile  # For writing NumPy array to GeoTIFF file.
from tifffile import imread, imwrite

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('mask_generation')

def initialize_earth_engine():
    """Authenticate and initialize Earth Engine with user-provided project ID."""
    project = getpass.getpass('Enter your EE Project ID: ').strip()
    if not project:
        raise ValueError('Earth Engine project ID is required.')

    try:
        ee.Initialize(project=project)
        logger.info('Earth Engine initialized with existing credentials.')
    except Exception:
        logger.info('Earth Engine authentication required; opening auth flow...')
        ee.Authenticate()
        ee.Initialize(project=project)
        logger.info('Earth Engine authenticated and initialized successfully.')
    return project

project_id = initialize_earth_engine()

if IN_COLAB and drive is not None:
    try:
        drive.mount('/content/gdrive/My Drive')
    except Exception as exc:
        logger.warning('Google Drive mount failed: %s', exc)
else:
    logger.info('Not running in Colab; skipping Google Drive mount.')

Mounted at /content/gdrive


In [ ]:
def load_config() -> tuple[str, dict]:
    """Load config.yaml from likely locations and validate required keys.
    Returns:
        A tuple containing the path to the config file and the loaded config dictionary.
    Raises:
        FileNotFoundError: If no config.yaml file is found in expected locations.
        ValueError: If the config file contains invalid YAML.
        KeyError: If required keys are missing from the config.
    """
    candidates = []
    if IN_COLAB:
        candidates.append(Path('/content/gdrive/My Drive/config.yaml'))

    # Local fallbacks for running outside Colab.
    candidates.extend([
        Path.cwd() / 'config.yaml',
        Path.cwd() / 'config' / 'config.yaml',
        Path.cwd().parent / 'config.yaml',
        Path.cwd().parent / 'config' / 'config.yaml'
    ])

    checked = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.add(str(candidate))

        if not candidate.exists():
            continue

        try:
            with open(candidate, 'r') as f:
                cfg = yaml.safe_load(f) or {}
        except yaml.YAMLError as exc:
            raise ValueError(f'Invalid YAML in config file: {candidate}') from exc

        required_keys = ['polygon', 'date_ranges', 'thresholds', 'paths']
        missing = [key for key in required_keys if key not in cfg]
        if missing:
            raise KeyError(
                f'Missing required config keys in {candidate}: {missing}'
            )

        return str(candidate), cfg

    raise FileNotFoundError(
        'Could not find config.yaml in expected locations. '
        'Update the path or place config.yaml in Drive/local config folder.'
    )

config_path, config = load_config()
logger.info('Loaded config from %s', config_path)

### Create EE Image collection

Spectral bands are a necessary component of geospatial analysis. Bands B2, B3, and B4 are essential for both RGB visualization and water quality algorithms. B8, the near-infrared band is critical for cloud detection and water/land discrimination. B11 and B12 are the shortwave infrared bands and provide atmospheric information excellent for cloud detection.

In [ ]:
# B1 --> Aerosols (60m)
# B2 --> Blue (10m)
# B3 --> Green (10m)
# B4 --> Red (10m)
# B5 --> Red Edge 1 (20m)
# B6 --> Red Edge 2 (20m)
# B7 --> Red Edge 3 (20m)
# B8 --> NIR (10m)
# B8A --> Red Edge 4 (20m)
# B9 --> Water vapor (60m)
# B11 --> SWIR 1 (20m)
# B12 --> SWIR 2 (20m)

# Select most optimal band combination for water quality analysis.
# water_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

# Subset of bands to be resampled
not_10m = ['B1', 'B5', 'B6', 'B7', 'B8A', 'B9', 'B11', 'B12']

# Select all bands
all_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

logger.info('Bands selected for processing: %s', all_bands)

The time period for this analysis includes selected dates from 2020 to 2025, and span the warmest months of the year, April through October. Since harmful algal blooms (HABs) thrive in warm, sunlit conditions, this time of year makes the most sense. This research work commenced in early 2023 and concluded in August 2024. The goal at the time was to study the last three years of change for the area of interest. Once I discovered the absence of the QA60 band from February 2022 through February 2024 due to changes in the processing pipeline, the selected dates were adjusted accordingly.

The cloud filter parameter has been initialized to 90 for this project, for the purposes of allowing a greater selection of data (the area is often cloudy and this generates a less exclusive selection), and to allow more cloudy scenes for a more thorough comparison.

After visualizing Sandusky Bay in the Copernicus Browser, a polygon was determined as the most suitable Earth Engine geometry for this region, and coordinates were pulled directly from this resource. Here, EPSG:4326 is the coordinate reference system assumed for projection, the standard for latitude and longitude based on WGS84.

In [ ]:
# Define area of interest: Sandusky Bay, Ohio.
AOI = ee.Geometry.Polygon(config['polygon'])
logger.info('Area of Interest defined with %d vertices.', len(config['polygon']))

# Define date ranges for image collection as list of tuples (start, end).
# Start and end dates should be in 'YYYY-MM-DD' format. End date is exclusive.   
date_ranges = [(dr['start'], dr['end']) for dr in config['date_ranges']]
logger.info('Date ranges defined: %s', date_ranges)

# Define the center point for plotting and metadata purposes.
# Coordinates for Folium maps are [latitude, longitude] (reverse of GEE).
center = AOI.centroid(10).coordinates().reverse().getInfo()

# Maximum cloud cover percent allowed in image collection.
CLOUD_FILTER = config['thresholds']['cloud_filter']
logger.info('Cloud cover filter set to %d%%', CLOUD_FILTER)

### Define reusable functions

#### Visualization settings and implementation for cloud mask and composite images

In [ ]:
# Define a method for displaying Earth Engine image tiles to a folium map.

def add_ee_layer(self, ee_image_object, vis_params, name, show=True, opacity=1, min_zoom=0) -> None:
    """Add a given EE image to a folium map as a tile layer.
    Args:
        self: The folium Map object to which the layer will be added.
        ee_image_object: An ee.Image object representing the Earth Engine image to display.
        vis_params: A dictionary of visualization parameters for styling the image layer.
        name: A string name for the layer, which will appear in the map legend.
        show: A boolean indicating whether the layer should be visible by default (default True).
        opacity: A float between 0 and 1 representing the opacity of the layer (default 1).
        min_zoom: An integer representing the minimum zoom level at which the layer will be visible (default 0).
    """
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
        name=name,
        show=show,
        opacity=opacity,
        min_zoom=min_zoom,
        overlay=True,
        control=True
        ).add_to(self)

# Add the Earth Engine layer method to folium.
folium.Map.add_ee_layer = add_ee_layer

In [ ]:
# Visualize cloud mask components layered over the area of interest.

def display_cloud_mask(col, mask) -> None:
    """Display cloud mask components over the area of interest.
    Args:
        col: An ee.ImageCollection object representing the image collection.
        mask: A string representing the name of the mask band to display.
    """
    # Mosaic the image collection.
    img = col.mosaic()

    # Subset layers and prepare them for display.
    cloudmask = img.select(mask).selfMask()

    # Create a folium map object.
    m = folium.Map(location=center, zoom_start=12)

    # Add layers to the folium map.
    m.add_ee_layer(img, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2500, 
                'gamma': 1.1},'S2 image', True, 1, 9)
    m.add_ee_layer(cloudmask, {'palette': 'orange'},'cloudmask', True, 0.5, 9)

    # Add a layer control panel to the map.
    m.add_child(folium.LayerControl())

    # Display the map.
    display(m)

In [ ]:
# Use with export function to visualize the cloud mask in black and white (RGB).

def cloud_mask_vis(image) -> ee.Image:
  """Visualize the cloud mask in black and white (RGB).
  Args:
      image: An ee.Image object representing the cloud mask.
  Returns:
      An ee.Image object with the cloud mask visualized in black and white.
  """
  # Define the range of pixel values that will be mapped to 0-255.
  vis = image.visualize(**{
    'palette': ['black', 'white'],
    "min": 0,     # Pixel values at or below this will be displayed as black.
    "max": 0.4    # Pixel values at or above this will be displayed as white.
  })
  return vis

#### Export EE image to Google Drive

In [ ]:
# Export to a folder in Google Drive. If raw mask is used as input for
# the image parameter, the output will be binary (values strictly 0 or 1). Else,
# if the cloud_mask_vis function is used with the mask, the output will contain values from 0 to 255.

def export_image(
    mask,
    description,
    folder='Results',
    region=AOI,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1_000_000_000,
    timeout_minutes=120,
    poll_seconds=60,
    raise_on_error=False
) -> dict:
    """Export an Earth Engine image and monitor task state with timeout handling.
    Args:
        mask: An ee.Image object representing the image to export.
        description: A string description for the export task, used as the filename.
        folder: A string representing the Google Drive folder to export to (default 'Results').
        region: An ee.Geometry object defining the export region (default AOI).
        scale: An integer representing the resolution in meters per pixel (default 10).
        crs: A string representing the coordinate reference system for export (default 'EPSG:4326').
        maxPixels: An integer representing the maximum number of pixels allowed in the export (default 1 billion).
        timeout_minutes: An integer representing the maximum time to wait for export completion in minutes (default 120).
        poll_seconds: An integer representing the interval in seconds to check the export task status (default 60).
        raise_on_error: A boolean indicating whether to raise exceptions on errors (default False). 
                        If False, errors will be logged and returned in the status dictionary.
    Returns:
        A dictionary containing the final status of the export task, including 'state' and any 'error_message' if applicable.
    Raises:
        RuntimeError: If the export task fails and raise_on_error is True.
        TimeoutError: If the export task times out and raise_on_error is True.
    Note:
        - The function starts an export task and continuously polls its status until it reaches 
        a terminal state ('COMPLETED', 'FAILED', 'CANCELLED') or exceeds the specified timeout.
        - If the task fails or is cancelled, the error message from the task status will be 
        logged and included in the returned status dictionary. If raise_on_error is True, 
        a RuntimeError will be raised with the error details.
        - If the task exceeds the timeout, it will attempt to cancel the task and return 
        a status indicating a timeout. If raise_on_error is True, a TimeoutError will be raised.
    """
    try:
        task = ee.batch.Export.image.toDrive(
            image=mask,
            description=description,    # Filename.
            # The Google Drive Folder that the export will reside in. Note:
            # (a) if the folder name exists at any level, the output is written to it,
            # (b) if duplicate folder names exist, output is written to the most recently modified folder,
            # (c) if the folder name does not exist, a new folder will be created at the root, and
            # (d) folder names with separators (e.g. 'path/to/file') are interpreted as literal strings,
            # not system paths. Defaults to Drive root.
            folder=folder,
            region=region,
            scale=scale,    # Resolution in meters per pixel. Defaults to 1000.
            crs=crs,      # Default is global GCS; EPSG:32617 is specific to Ohio.
            maxPixels=maxPixels
        )
        task.start()
        logger.info('Export started: %s (folder=%s)', description, folder)
    except Exception as exc:
        logger.error('Failed to start export for %s: %s', description, exc)
        if raise_on_error:
            raise
        return {'state': 'START_FAILED', 'error_message': str(exc)}

    start_time = time.time()
    last_state = None
    terminal_states = {'COMPLETED', 'FAILED', 'CANCELLED'}

    while True:
        try:
            status = task.status()
        except Exception as exc:
            logger.error('Failed to read task status for %s: %s', description, exc)
            if raise_on_error:
                raise
            return {'state': 'STATUS_ERROR', 'error_message': str(exc)}

        state = status.get('state', 'UNKNOWN')
        if state != last_state:
            logger.info('Task %s state: %s', description, state)
            last_state = state

        if state in terminal_states:
            if state == 'COMPLETED':
                logger.info('Export completed: %s', description)
            else:
                logger.error(
                    'Export ended with state %s for %s. Details: %s',
                    state,
                    description,
                    status.get('error_message', 'No error message provided.')
                )
                if raise_on_error:
                    raise RuntimeError(
                        f"Export failed for {description}: {status.get('error_message', state)}"
                    )
            return status

        elapsed_minutes = (time.time() - start_time) / 60
        if elapsed_minutes > timeout_minutes:
            logger.error('Export timed out after %.1f minutes: %s', elapsed_minutes, description)
            try:
                task.cancel()
                logger.warning('Timed-out task cancelled: %s', description)
            except Exception as exc:
                logger.warning('Could not cancel task %s: %s', description, exc)

            timeout_status = {'state': 'TIMEOUT', 'description': description}
            if raise_on_error:
                raise TimeoutError(f'Export timed out for {description}.')
            return timeout_status

        time.sleep(poll_seconds)

#### Preprocess cloud masks for each algorithm by upsampling bands

Reflectance bands and all ancillary bands for all algorithms are resampled (upsampled) to 10m resolution. This is important for avoiding alignment mismatches, i.e., ensuring a fair comparison. Although it requires larger storage and compute, upsampling is preferable to downsampling in this study for two reasons. Spatial detail is crucial for small water features and cloud edges, and 10m resolution allows for preservation of such detail. Also, s2cloudless and Cloud Score+ expect 10 m bands for the visible/NIR inputs, and downsampling may change their behaviour in ways unrelated to the algorithm. Bilinear interpolation is used for continuous reflectance and probability layers to avoid blocky artifacts, while nearest-neighbor is needed for categorical/classification layers to avoid creating mixed fractional classes that break thresholds/logic.

In [ ]:
# --- Core preprocessing for reflectance bands ---
def preprocess_reflectance(img) -> ee.Image:
    """
    Resamples all reflectance bands to 10 m.
    Args:        
        img: An ee.Image object containing Sentinel-2 reflectance bands.
    Returns:        
        An ee.Image object with all reflectance bands resampled to 10 m resolution.
    """
    refl_10m = (img.select(not_10m)
                .resample('bilinear')
                .reproject(crs='EPSG:4326', scale=10))
    return img.addBands(refl_10m, overwrite=True)

# --- Hybrid method preprocessing ---
def preprocess_hybrid(img) -> ee.Image:
    """
    Preprocesses an image using the hybrid method.
    Args:
        img: An ee.Image object containing Sentinel-2 bands and ancillary data.
    Returns:
        An ee.Image object with all bands resampled to 10 m resolution and clipped to the buffered AOI.
    """
    aoi_buffered = AOI.buffer(20)
    # Start with reflectance bands.
    img_out = preprocess_reflectance(img)

    # Resample ancillary bands to 10m.
    cldprb = (img.select('MSK_CLDPRB')
                .resample('bilinear')
                .reproject(crs='EPSG:4326', scale=10)
                .clamp(0, 1))

    classi = (img.select(['MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS'])
                # Nearest-neighbor is default.
                .resample()
                .reproject(crs='EPSG:4326', scale=10))

    # Add the upsampled ancillary bands.
    img_out = img_out.addBands([cldprb, classi], overwrite=True)

    # Clip after all bands are added.
    img_out = img_out.clip(aoi_buffered)

    return img_out

# --- s2cloudless preprocessing ---
def preprocess_s2cloudless(img) -> ee.Image:
    """ Preprocesses an image using the s2cloudless method.
    Args:
        img: An ee.Image object containing Sentinel-2 bands and ancillary data.
    Returns:
        An ee.Image object with all bands resampled to 10 m resolution and clipped to the buffered AOI.
    """
    img_out = preprocess_reflectance(img)

    scl = (img.select('SCL')
            .resample()    # Nearest-neighbor approach.
            .reproject(crs='EPSG:4326', scale=10))

    img_out = img_out.addBands(scl, overwrite=True)

    return img_out.clip(AOI)

# Resample reflectance bands and clip each image to AOI.
# For use in Cloud Score + algorithm and IRIS input.
def preprocess_reflectance_with_clip(img) -> ee.Image:
    """
    Resamples reflectance bands and clips the image to the AOI.
    Args:
        img: An ee.Image object containing Sentinel-2 reflectance bands.
    Returns:
        An ee.Image object with all reflectance bands resampled to 10 m resolution and clipped to the AOI.
    """
    img_out = preprocess_reflectance(img)
    return img_out.clip(AOI)

## Masking Algorithms

### Probabilistic Cloud Mask and Classification Masks Hybrid

In the COPERNICUS/S2_SR_HARMONIZED dataset, several QA and mask bands are provided. All are derived from the European Space Agency's Sen2Cor processor. QA60 is the standard quality assurance band embedded in all Sentinel-2 L1C products. Essentially, it is a binary classifier and bitmask band. The band has been absent since 2022-01-25 due to changes in the processing pipeline. Introduced to address this issue, the Sentinel-2 Harmonized data collection includes QA60 bands generated from the MSK_CLASSI cloud classification bands. This is the simplest algorithm to implement and is included in nearly every cloud masking algorithm comparison study. The inability to fine-tune this method yields a high omission error, an issue with compounding effects in high-level analyses. For this reason, I decided to test a hybrid approach combining the benefits of both the cloud probability band (MSK_CLDPRB) and the classification bands (MSK_CLASSI), both of which are more accurate than QA60. The idea here is to reduce false positives and missed clouds, offering an approach that is more in line with the other two algorithms being compared.

#### Create and export the cloud mask

In [ ]:
# ------------------------------------------------------------
# Hybrid Approach (cloud probability and classification bands)
# ------------------------------------------------------------
def add_hybrid_cloud_mask(img) -> ee.Image:
    """
    Adds a hybrid cloud mask to the image.
    Args:
        img: An ee.Image object containing Sentinel-2 bands and ancillary data.
    Returns:
        An ee.Image object with an added hybrid cloud mask band.
    """
    cldprb = img.select('MSK_CLDPRB')
    opaque = img.select('MSK_CLASSI_OPAQUE')
    cirrus = img.select('MSK_CLASSI_CIRRUS')

    hybrid_threshold = config['thresholds']['hybrid']
    logger.info('Applying hybrid cloud mask with probability threshold: %.2f', hybrid_threshold)
    # Threshold the probability band and combine with classification bands (1=clouds).
    qa60_cloud_mask = cldprb.gt(hybrid_threshold).Or(opaque.eq(1)).Or(cirrus.eq(1))
    cloud_band = ee.Image(qa60_cloud_mask).rename('hybrid_mask')
    return img.addBands(cloud_band)

In [ ]:
for start_date, end_date in date_ranges:
    # Define and filter Sentinel-2 A/B Harmonized Surface Reflectance dataset.
    hybrid_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(AOI)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .select(all_bands + ['MSK_CLDPRB', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS'])
    )

    # Resample bands to 10m resolution for all images in the image collection.
    hybrid_collection = hybrid_col.map(preprocess_hybrid)
    #print('Hybrid bands:', hybrid_collection.first().bandNames().getInfo())

    # Apply the hybrid cloud mask to every image in the upsampled collection.
    upsampled_coll_with_mask = hybrid_collection.map(add_hybrid_cloud_mask)

    # Choose a single image using mosaic compositing.
    hybrid_mask = upsampled_coll_with_mask.select('hybrid_mask').mosaic()

    # Display cloud mask as a layer in a Folium map.
    display_cloud_mask(upsampled_coll_with_mask, 'hybrid_mask')

    # Export cloud mask to Google Drive.
    export_image(hybrid_mask, start_date.replace('-', '')[:-2], 'hybrid')

logger.info('Hybrid cloud masking and export completed for all date ranges.')

### S2Cloudless
Developed by a company called Sinergise, s2cloudless is available on Sentinel Hub and in Google Earth Engine. A supervised machine learning method is used, specifically a LightGBM decision tree model, along with the following spectral bands: B1, B2, B4, B5, B8, B8A, B9, B10, B11, B12 (Sentinel-2 Level-1C TOA surface reflectance values). Trained on a global distribution of 13 million scenes, including cloud masks from MAJA, s2cloudless uses a 10m spatial resolution, 160m buffer, and is mono-temporal. The output is a cloud probability map. In cloud probability, higher values are more likely to be clouds or highly reflective surfaces.

The algorithm is implemented as follows. Start by building a collection that combines the harmonized Sentinel-2 surface reflectance and Sentinel-2 cloud probability collections as one. They both need to have similar filters with the same bounds and date. They are joined on the “system:index” property. The result is essentially a copy of the surface reflectance collection with a new property with a value corresponding to the s2cloudless image. The next step is to define the cloud mask component function, for which the s2cloudless probability layer and derived cloud mask are added as bands to an S2 surface reflectance image input. The mask is created by thresholding the probability band.

s2cloudless was selected for comparison due to its reputation as one of the best cloud detection algorithms, low computational requirements, accessability and strong documentation. The following section uses a modified version of the sample code presented in the Python tutorials section of Google Earth Engine, found here: https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless.


In [ ]:
# Initialize parameters and join filtered collections.

CLD_PRB_THRESH = config['thresholds']['s2cloudless']['cloud_probability']   # Cloud probability (%); values greater than are considered cloud (int)
NIR_DRK_THRESH = config['thresholds']['s2cloudless']['nir_darkness']   # Near-infrared reflectance; values less than are considered potential cloud shadow (float)
CLD_PRJ_DIST = config['thresholds']['s2cloudless']['cloud_projection_distance']      # Maximum distance (km) to search for cloud shadows from cloud edges (float)
BUFFER = config['thresholds']['s2cloudless']['buffer']     # Distance (m) to dilate the edge of cloud-identified objects.

#### Add the s2cloudless probability layer and derived cloud mask as bands to an S2 SR image input.

In [ ]:
def add_cloud_bands(img) -> ee.Image:
    """
    Adds cloud probability and cloud mask bands to the image.
    Args:
        img: An ee.Image object containing Sentinel-2 bands and ancillary data.
    Returns:
        An ee.Image object with added cloud probability and cloud mask bands.
    """
    # Retrieve a pre-computed cloud probability map, which contains values indicating 
    # the likelihood of a cloudy pixel. 
    # Essentially, create a cloud probability layer from the s2cloudless output band.
    cld_prb_band = ee.Image(img.get('s2cloudless')).select('probability')

    # Everything above the cloud probability threshold is considered a cloud (1), else clear (0).
    is_cloud = cld_prb_band.gt(CLD_PRB_THRESH).rename('clouds')
    logger.info('Applying s2cloudless cloud mask with probability threshold: %d%%', CLD_PRB_THRESH)

    # Return the image object with the two bands (cloud probability layer and cloud mask) added.
    return img.addBands(ee.Image([cld_prb_band, is_cloud]))

In [ ]:
def add_shadow_bands(img) -> ee.Image:
    """
    Adds shadow identification bands to the image.
    Args:
        img: An ee.Image object containing Sentinel-2 bands and ancillary data.
    Returns:
        An ee.Image object with added shadow identification bands.
    """
    # Identify potentially dark (non-water) pixels that may be cloud shadows using a 
    # NIR (band B8) multiplied by a common scale factor, and creates a binary mask where    
    # 1 indicates potential shadow pixels and 0 indicates non-shadow pixels (water pixels are masked out).
    SR_BAND_SCALE = 1e4
    dark_pixels = (img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE)
                    .multiply(img.select('SCL').neq(6)).rename('dark_pixels'))

    # The solar azimuth angle is the horizontal angle with respect to north of the Sun's position. 
    # It defines the Sun's relative direction along the local horizon. Here, we calculate 
    # the shadow azimuth by subtracting the single average azimuth value calculated for 
    # the entire image scene or tile from 90 degrees. A UTM projection is assumed.
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))

    # Project potential cloud shadows by selecting the cloud mask and applying the 
    # directionalDistanceTransform function, which computes the distance to the nearest 
    # cloud pixel in the direction specified by the previously computed shadow azimuth angle, 
    # to a maximum distance (user-defined float variable multiplied by 10). The result is 
    # reprojected to match the image's projection and scale (scale is 10 for this project). 
    # Areas beyond the projection distance are then masked out.
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 10})
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    # Use the intersection of the projected shadow areas and dark non-water pixels to 
    # identify cloud shadows. Their product results in a band where a value of 1 
    # indicates dark pixels that fall within a projected cloud shadow area. 
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')
    logger.info('Applied cloud shadow identification with NIR darkness threshold: '
    '%f and projection distance: %f km', NIR_DRK_THRESH, CLD_PRJ_DIST)

    # Return the input image with the three added bands.
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

In [ ]:
# Build the cloud mask by combining the cloud and shadow component bands,
# and applying morphological operations to clean up the mask.

def cld_mask(img) -> ee.Image:
    """
    Builds the cloud mask by combining the cloud and shadow component bands,
    and applying morphological operations to clean up the mask.
    Args:
        img: An ee.Image object containing Sentinel-2 bands and ancillary data.
    Returns:
        An ee.Image object with an added cloud-shadow mask band.
    """
    # Add bands to help with cloud detection.
    img_cloud = add_cloud_bands(img)

    # Add bands to help with cloud shadows detection.
    img_cloud_shadow = add_shadow_bands(img_cloud)

    # Subset cloud and shadow image bands and perform pixel-wise addition. Set their value to 1.
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)

    # Refine the combined cloud-shadow mask to reduce noise and remove where possible. 
    # The BUFFER variable can be fine-tuned to capture cloud shadows as accurately as possible.
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER*2/20)
        # Reprojection is necessary after focal operations to prevent scale and CRS information loss, 
        # which can cause issues in downstream processing and export. Scale is 10m to match the 
        # original image resolution and provide more precise cloud edges for masking
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 10})
        .rename('cloudmask'))
    logger.info('Applied morphological operations to cloud-shadow mask with buffer: %d m', BUFFER)

    # Return the input image with the refined (binary cloud-shadow mask) band added.
    return img_cloud_shadow.addBands(is_cld_shdw)

#### Build the collection. Visualize and export the cloud mask.

In [ ]:
for start_date, end_date in date_ranges:
    # Resample bands to 10m resolution for all images in the image collection.
    s2_sr_col_plus_scl = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(AOI)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .select(all_bands + ['SCL'])
    )

    # Resample reflectance and ancillary bands to 10m resolution for all images in collection.
    upsampled_s2_collection = s2_sr_col_plus_scl.map(preprocess_s2cloudless)
    print('s2cloudless bands:', upsampled_s2_collection.first().bandNames().getInfo())

    # Import and filter s2cloudless.
    s2cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
        .filterBounds(AOI)
        .filterDate(start_date, end_date))

    # Join probability and SR collections on matching property value and name it.
    s2cloudless = ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': upsampled_s2_collection,
        'secondary': s2cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index',
            'rightField': 'system:index'
        })
    }))

    # --- Display image and mask component layers ---
    # Apply cloud mask to every image in the collection.
    s2cloudless_final_mask = s2cloudless.map(cld_mask)

    display_cloud_mask(s2cloudless_final_mask, 'cloudmask')

    # --- Export raster image file of cloud mask ---
    # Loading final cloud mask (cloud and shadows).
    s2cloudless_img = s2cloudless_final_mask.mosaic()
    s2cloudless_mask = s2cloudless_img.select('cloudmask')

    # Export the cloud mask to Google Drive.
    export_image(s2cloudless_mask, start_date.replace('-', '')[:-2], 's2cloudless')

logger.info('s2cloudless cloud masking and export completed for all date ranges.')


### Cloud Score +

Developed by a small group of researchers and geospatial data scientists, Cloud Score + uses a weakly supervised deep learning approach for individual pixel QA. It is generated from the Sentinel-2 L1C, and works equally well on L2A. “Weakly supervised” means that a very small set of manually annotated images with assigned “meaning” to scores, known as an atmospheric similarity index metric (or measure), was augmented by synthetic ones. A bootstrapping procedure leveraging millions of automatically-mined training samples via short video clips were informed by this measure to assign the per-pixel QA (usability) scores. The algorithm was trained with clear references, mean, standard deviation, image, terrain, and metadata, and the ASIM scores are used to create masks by thresholding. Thresholding allows for finetuning and improved results.

The harmonized Sentinel-2 image collection is linked with Cloud Score + using an Earth Engine method called linkCollection(), a single line join to join bands to ee.Image or ee.ImageCollection. Cloud score is a 2-band image in the collection. The two QA bands are cs and cs_cdf. cs can be thought of as a more instantaneous atmospheric similarity score, while cs_cdf captures an expectation of the estimated score through time. The cs_cdf band is selected for this cloud mask experiment because it is less sensitive to thin haze around clouds, offering higher precision, and is therefore more comparable to the other two algorithms in terms of performance. Essentially, this band gives a likelihood of cloud and a tighter, neater boundary.

#### Build collection and generate cloud mask

In [ ]:
# Threshold for masking and QA band name for Cloud Score + method.

CLEAR_THRESHOLD = config['thresholds']['cloudscoreplus']   # CloudScore+ values above this threshold are masked as clouds.
QA_BAND = 'cs_cdf'
logger.info('Cloud Score + threshold set to %d', CLEAR_THRESHOLD)
logger.info('Cloud Score + QA band selected: %s', QA_BAND)

In [ ]:
# For visualization, the cloud mask needs to be applied to every image in the collection.

def apply_csplus_mask(img) -> ee.Image:
    """
    Applies the CS+ cloud mask to the image.
    Args:
        img: An ee.Image object containing Sentinel-2 bands and ancillary data.
    Returns:
        An ee.Image object with an added CS+ cloud mask band.
    """
    csplus_mask = img.select(QA_BAND).gte(CLEAR_THRESHOLD).Not().rename('csplus_mask')
    return img.addBands(csplus_mask)

#### Visualize and export cloud mask for selected dates

In [ ]:
for start_date, end_date in date_ranges:
  # Import and filter Sentinel-2 A/B Harmonized Surface Reflectance dataset.
  s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(AOI)
      .filterDate(start_date, end_date)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
      .select(all_bands)
      )

  # Resample bands to 10m resolution for all images in the image collection.
  upsampled_csplus_col = s2_sr_col.map(preprocess_reflectance_with_clip)
  #print('Cloud Score + bands:', upsampled_csplus_col.first().bandNames().getInfo())

  # Build Cloud Score + collection with 10m cs_cdf QA band.
  csplus_col = upsampled_csplus_col.linkCollection(
    ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'),QA_BAND)

  # Apply the mask to every image in the Cloud Score+ collection
  # and visualize as a layer on a Folium map.
  csplus = csplus_col.map(apply_csplus_mask)
  display_cloud_mask(csplus, 'csplus_mask')

  # Initiate a cloud mask object for export to Google Drive.
  image = csplus_col.mosaic()
  csplus_cloud_mask = image.select(QA_BAND).gte(CLEAR_THRESHOLD)
  csplus_cloud_mask_export = csplus_cloud_mask.Not()

  export_image(csplus_cloud_mask_export, start_date.replace('-', '')[:-2], 'cloudscoreplus')

logger.info('Cloud Score + cloud masking and export completed for all date ranges.')

## Generating inputs for IRIS program

IRIS, an active learning software tool by ESA-PhiLab, stands for Intelligently Reinforced Image Segmentation. Cesar Aybar and Ali Francis developed this tool to "accelerate the creation of machine learning training datasets for Earth Observation", as stated on their GitHub repository. Essentially, the software runs locally as a Flask app with a single configuration file to segment multi-spectral and geospatial imagery. Setup and customization are fairly simple. The segmentation process is semi-automatic, as pixels are labeled manually with the support of AI, i.e. a gradient boosted decision tree. Especially useful for cloud segmentation, the data can be transformed into a binary mask containing "clear" and "cloud" classes, or further separated into "cloud shadows", "thick cloud", and "thin cloud". Its purpose in this project is to generate high-quality labeled reference masks necessary for algorithm comparison. No reference masks had been previously generated for Sandusky Bay. Alternative methods were either not feasible, did not offer the same level of quality, or were simply not as convenient.

Primary input is a multi-page TIFF image file containing raw reflectance values of type 'float32'. Thumbnails and metadata are optional, but were used as data validation for each scene. The masks were output in .tif format, encoded as RGB, and scored using F1, as the goal was to achieve balanced performance. A total of 15 scenes of the same area of interest (Sandusky Bay) were segmented, representing various levels of cloudiness, including a variety of cloud types and shadows.

### Initial GeoTIFF image export

Generate the intial GeoTIFF file of our AOI, to use as input for the final .tif file containing an array of raw reflectance values, a necessary component for the IRIS program to function. We are using GeoTIFF to preserve metadata.

In [ ]:
for start, end in date_ranges:
    # Import and filter Sentinel-2 A/B Harmonized Surface Reflectance dataset.
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(AOI)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .select(all_bands))

    # Resample all reflectance bands and clip to AOI. Apply to every image in the collection.
    iris_col = s2_sr_col.map(preprocess_reflectance_with_clip)
    #print('Bands:', iris_col.first().bandNames().getInfo())

    # Create the image by compositing all the images in the collection (i.e.,
    # reduces collection to a single image). This method works best for this region
    # to provide complete coverage, since it lies between two areas captured by the satellite.
    iris_img = iris_col.mosaic()

    # Export RGB GeoTIFF file to Google Drive.
    export_image(iris_img, f'GeoTIFF_{start}', 'IRIS')

logger.info('IRIS image export completed for all date ranges.')

### Export TIFF containing array of raw reflectance values

Using the previously exported GeoTIFF file, a multi-page
32-bit float grayscale .tif file containing the raw reflectance values is created for use as input in the IRIS software, similar to the one used in the demo.

In [ ]:
# Write a 3-dimensional NumPy array to a multi-page grayscale TIFF file.
# Wait until initial GeoTIFF file export is complete to run this section.

for start, end in date_ranges:
    try:
        # Creates a 32-bit float image containing raw reflectance values (not scaled to 0-255).
        input_file = Path(config['paths']['iris_root']) / f'GeoTIFF_{start}.tif'
        output_path = Path(config['paths']['iris_root']) / f'raw_{start}.tif'

        if not input_file.exists():
            logger.warning('Skipping %s because input file is missing: %s', start, input_file)
            continue

        data = imread(input_file)
        logger.info('Loaded %s with shape %s and dtype %s', input_file.name, data.shape, data.dtype)

        data = data.astype('float32')

        # Save as a multi-page TIFF.
        output_path.parent.mkdir(parents=True, exist_ok=True)
        tifffile.imwrite(output_path, data, photometric='minisblack')
        logger.info('File saved to: %s', output_path)
    except Exception as exc:
        logger.exception('Failed to generate raw TIFF for %s: %s', start, exc)

### Optional PNG thumbnail image export

In [ ]:
for start, end in date_ranges:
    try:
        # Import and filter Sentinel-2 A/B Harmonized Surface Reflectance dataset.
        s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(AOI)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
            .select(all_bands))

        iris_col = s2_sr_col.map(preprocess_reflectance_with_clip)
        iris_img = iris_col.mosaic()

        thumbnail_url = iris_img.getThumbURL({
            'min': 0,                 # Pixel values at or below this will be displayed as black.
            'max': 1500,             # Pixel values at or above this will be displayed as white.
            'region': AOI,
            'format': 'png',
            'gamma': 0.8,     # Gamma >1 lightens the image; <1 darkens it.
            'bands': ['B4','B3','B2'],
            'dimensions': 512
        })

        logger.info('Thumbnail URL generated for %s', start)

        # Download the thumbnail to the IRIS folder in Google Drive.
        thumbnail_path = Path(config['paths']['iris_root']) / f'thumbnail_{start}.png'
        thumbnail_path.parent.mkdir(parents=True, exist_ok=True)

        response = requests.get(thumbnail_url, timeout=60)
        response.raise_for_status()

        with open(thumbnail_path, 'wb') as f:
            f.write(response.content)

        logger.info('Thumbnail saved to: %s', thumbnail_path)
    except requests.RequestException as exc:
        logger.error('Network/download error while exporting thumbnail for %s: %s', start, exc)
    except Exception as exc:
        logger.exception('Failed to export thumbnail for %s: %s', start, exc)

### Optional metadata export

In [ ]:
for start, end in date_ranges:
    try:
        # Create metadata.
        metadata = {
            'image_id': start,
            'region': AOI.getInfo(),
            'location': center,
            'bands': all_bands,
            'crs': 'EPSG:4326',
            'start_date': start,
            'end_date': end,
            'cloud_filter': CLOUD_FILTER
        }

        # Export metadata to JSON file in Google Drive.
        metadata_path = Path(config['paths']['iris_root']) / f'metadata_{start}.json'
        metadata_path.parent.mkdir(parents=True, exist_ok=True)

        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        logger.info('Metadata saved to: %s', metadata_path)
    except Exception as exc:
        logger.exception('Failed to write metadata for %s: %s', start, exc)